
<a href="https://colab.research.google.com/github/adenikeadewumi/python-ml-WIEOAU/blob/main/11_pandas/11_pandas.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Module 11 — Pandas: Data Manipulation

**Learning Objectives:** Series, DataFrame, selecting data, cleaning, groupby, merging, reading files

**Estimated time:** 75–90 minutes

---

## 11.1 What Is Pandas?

**The big picture:**
Pandas is Python's answer to Excel — but programmable, scalable to millions of rows, and integrated directly with your ML pipeline. If you work with tabular data (rows and columns), pandas is the tool you will use every single day.

**Two core objects:**
1. **Series** — a single column of data with a label (index) for each row. Think of it as a labelled 1D array.
2. **DataFrame** — a table with multiple named columns. Each column is a Series. Think of it as a spreadsheet or SQL table.

**Why pandas over plain NumPy?**
NumPy is excellent for uniform numerical arrays (like pixel values in an image). Real-world datasets are messier — they have column names, mixed types (numbers, text, dates), missing values, and categorical data. Pandas handles all of this elegantly.

**The pandas workflow in ML:**
```
Load data -> Explore -> Clean -> Transform -> Feed to model
```
Pandas handles every step except the last, which is scikit-learn/TensorFlow.

## 11.2 Series — A Labelled Column

**What is a Series?**
A Series is a one-dimensional array where each element has a label (the index). Unlike a plain list or NumPy array, a Series lets you look up values by label, not just by position. The index can be integers, strings, dates — anything hashable.

**Series operations are vectorised:**
Just like NumPy, arithmetic on a Series operates on every element at once. Comparisons return a boolean Series, which you can use to filter rows.

In [ ]:
import pandas as pd
import numpy as np

# Creating a Series
temps = pd.Series([22, 18, 25, 20, 17],
                  index=["Mon", "Tue", "Wed", "Thu", "Fri"])
print(temps)
print()

# Accessing by label (like a dict)
print("Wednesday:", temps["Wed"])

# Accessing by position (like a list)
print("First day:", temps.iloc[0])

# Boolean indexing — filter values
print("Days above 20 degrees:")
print(temps[temps > 20])

# Arithmetic — applied to every element
print("Temps in Fahrenheit:")
print(temps * 9/5 + 32)

# Summary stats
print(f"Mean: {temps.mean():.1f}, Max: {temps.max()}, Min: {temps.min()}")

## 11.3 DataFrame — The Workhorse

**What is a DataFrame?**
A DataFrame is a 2D table — a collection of Series that all share the same index. Each column has a name and can have a different dtype. This maps directly to what you see in a spreadsheet or a database table.

**The index:**
Every DataFrame has a row index. By default it is 0, 1, 2, ... but you can set it to anything meaningful (student IDs, timestamps, etc.). The index does not count as a column — it is the row label.

**Exploring a new DataFrame — always start here:**
Before doing anything with data, you should always run: `.shape`, `.dtypes`, `.head()`, `.describe()`, and `.isnull().sum()`. This tells you what you are working with before you start transforming anything.

In [ ]:
import pandas as pd
import numpy as np

# Create a DataFrame from a dict of lists
# Keys become column names; lists become column values
data = {
    "name":       ["Alice", "Bob", "Carol", "David", "Eve"],
    "age":        [25, 30, 28, 35, 22],
    "city":       ["London", "NYC", "London", "Paris", "NYC"],
    "salary":     [55000, 72000, 61000, 85000, 48000],
    "experience": [3, 8, 5, 12, 1],
}
df = pd.DataFrame(data)
print(df)
print()
print("Shape (rows, cols):", df.shape)
print("Column types:
", df.dtypes)

In [ ]:
# Exploration functions — run these on every new dataset
print("First 3 rows:")
print(df.head(3))
print()

print("Last 2 rows:")
print(df.tail(2))
print()

print("Summary statistics for numerical columns:")
print(df.describe().round(1))
print()

print("Column info (types + non-null counts):")
df.info()
print()

print("Missing values per column:")
print(df.isnull().sum())

## 11.4 Selecting Data

**Two primary ways to select data:**

1. **`df[column_name]`** — select one or more columns
2. **`df.loc[rows, columns]`** — select by **label** (row label + column name)
3. **`df.iloc[rows, columns]`** — select by **integer position** (0-based index)

**The key distinction between `.loc` and `.iloc`:**
- `.loc` uses the actual index labels. If your index is `["a", "b", "c"]`, then `df.loc["b"]` gets the row labelled "b".
- `.iloc` uses the position number regardless of labels. `df.iloc[1]` always gets the second row.

**Boolean filtering** is how you select rows based on data values — the equivalent of a SQL WHERE clause.

In [ ]:
import pandas as pd
data = {
    "name": ["Alice","Bob","Carol","David","Eve"],
    "age":  [25, 30, 28, 35, 22],
    "city": ["London","NYC","London","Paris","NYC"],
    "salary": [55000, 72000, 61000, 85000, 48000],
}
df = pd.DataFrame(data)

# Select a single column -> returns a Series
print("Names column:")
print(df["name"])
print()

# Select multiple columns -> returns a DataFrame
print("Name and salary:")
print(df[["name", "salary"]])
print()

# .loc — by label
print("Row 0 to 2, all columns:")
print(df.loc[0:2])   # note: .loc is INCLUSIVE of stop index

# .iloc — by position
print("First 3 rows, first 2 columns:")
print(df.iloc[:3, :2])

In [ ]:
import pandas as pd
data = {
    "name": ["Alice","Bob","Carol","David","Eve"],
    "age":  [25, 30, 28, 35, 22],
    "city": ["London","NYC","London","Paris","NYC"],
    "salary": [55000, 72000, 61000, 85000, 48000],
}
df = pd.DataFrame(data)

# Boolean filtering — most common way to filter rows

# Single condition
high_earners = df[df["salary"] > 60000]
print("High earners:")
print(high_earners)
print()

# Multiple conditions — use & (and) and | (or) with parentheses around each condition
london_high = df[(df["city"] == "London") & (df["salary"] > 55000)]
print("London high earners:")
print(london_high)
print()

# .isin() — filter where value is in a list
target_cities = df[df["city"].isin(["London", "Paris"])]
print("London or Paris employees:")
print(target_cities)

## 11.5 Data Cleaning

**Why data cleaning?**
Real-world data is dirty. It has missing values, typos, duplicate rows, outliers, wrong data types, and inconsistent formatting. A model trained on dirty data learns the dirt. Data cleaning is the highest-leverage work a data scientist does — a clean dataset with a simple model outperforms a dirty dataset with a complex model.

**The four most common cleaning tasks:**
1. **Missing values** — detect with `.isnull()`, handle by dropping or filling
2. **Duplicates** — detect with `.duplicated()`, remove with `.drop_duplicates()`
3. **Wrong dtypes** — fix with `.astype()` or `pd.to_datetime()`
4. **Outliers** — detect with `.describe()` and plots, handle by capping or removing

**Filling strategies for missing values:**
- Fill with a constant (zero, "Unknown")
- Fill with the mean/median of the column (median is better for skewed data)
- Forward-fill or back-fill (for time series — carry last known value forward)
- Fill per group (fill with the group's median, not the overall median)

In [ ]:
import pandas as pd
import numpy as np

# Create a messy dataset
df = pd.DataFrame({
    "name":   ["Alice", "Bob", None,    "David", "Alice"],
    "age":    [25,       30,   28,       200,     25],     # 200 is an outlier
    "salary": [55000,    None, 61000,    85000,   55000],  # Bob's salary missing
    "city":   ["London", "NYC","London", "Paris", "London"]
})
print("Raw messy data:")
print(df)
print()

# Step 1: check what we are dealing with
print("Missing values:")
print(df.isnull().sum())
print()
print("Duplicates:", df.duplicated().sum())

# Step 2: handle missing values
df["name"].fillna("Unknown", inplace=True)
df["salary"].fillna(df["salary"].median(), inplace=True)  # fill with median

# Step 3: remove duplicate rows
df.drop_duplicates(inplace=True)

# Step 4: handle outlier (age=200 is impossible)
print("
Age statistics before fixing:")
print(df["age"].describe())
df = df[df["age"] < 120]   # remove rows where age is unrealistic

print("
Cleaned data:")
print(df.reset_index(drop=True))

## 11.6 GroupBy — Split, Apply, Combine

**What is groupby?**
GroupBy is one of the most powerful pandas operations. It follows three steps:
1. **Split** — divide the DataFrame into groups based on one or more column values
2. **Apply** — run a function on each group independently
3. **Combine** — collect the results back into a single DataFrame

**This is equivalent to SQL's `GROUP BY` clause.** "What is the average salary per department?" — that is a groupby question.

**The `.agg()` method** lets you apply multiple aggregation functions at once, or different functions to different columns.

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "name":   ["Alice","Bob","Carol","David","Eve","Frank"],
    "dept":   ["Eng","HR","Eng","Finance","HR","Eng"],
    "salary": [72000, 55000, 68000, 82000, 59000, 65000],
    "score":  [88, 75, 92, 68, 81, 77],
    "years":  [4, 7, 3, 12, 5, 2],
})

# Basic groupby — mean salary per department
print("Average salary by department:")
print(df.groupby("dept")["salary"].mean().round(0))
print()

# Multiple aggregations at once
print("Salary stats by department:")
print(df.groupby("dept")["salary"].agg(["mean", "max", "min", "count"]).round(0))
print()

# Different aggregations for different columns
print("Department summary:")
summary = df.groupby("dept").agg(
    avg_salary=("salary", "mean"),
    max_score=("score", "max"),
    avg_years=("years", "mean"),
    headcount=("name", "count")
).round(1)
print(summary)

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name":   ["Alice","Bob","Carol","David","Eve","Frank"],
    "dept":   ["Eng","HR","Eng","Finance","HR","Eng"],
    "salary": [72000, 55000, 68000, 82000, 59000, 65000],
})

# transform() — returns a value for EVERY row aligned with the original index
# This is different from agg() which returns ONE row per group
# Use transform() when you want to compare each row to its group's statistic

df["dept_avg_salary"]  = df.groupby("dept")["salary"].transform("mean")
df["above_dept_avg"]   = df["salary"] > df["dept_avg_salary"]

print("Each employee vs their department average:")
print(df.to_string())

## 11.7 Merging DataFrames

**What is a merge?**
A merge combines two DataFrames based on a shared column (or index). This is exactly like SQL JOINs.

**The four types of joins:**
- **inner** — only rows that have a match in BOTH DataFrames (default)
- **left** — all rows from the left DataFrame, matched rows from right (unmatched = NaN)
- **right** — all rows from the right DataFrame, matched rows from left
- **outer** — all rows from BOTH DataFrames

**When do you need merges?**
Whenever your data lives in multiple tables. In a company database: employees table, departments table, salaries table. In ML: feature table + labels table. In analysis: this year's data + last year's data.

In [ ]:
import pandas as pd

employees = pd.DataFrame({
    "emp_id": [1, 2, 3, 4, 5],
    "name":   ["Alice", "Bob", "Carol", "David", "Eve"],
    "dept_id":[1, 2, 1, 3, 2]
})

departments = pd.DataFrame({
    "dept_id":  [1, 2, 3],
    "dept_name":["Engineering", "Marketing", "Finance"],
    "location": ["London", "NYC", "Paris"]
})

print("Employees:")
print(employees)
print("
Departments:")
print(departments)

# Inner join — only employees with a matching department
merged = employees.merge(departments, on="dept_id", how="left")
print("
Merged (left join):")
print(merged)

# Add a new employee with no department match
employees_new = employees.append(
    {"emp_id": 6, "name": "Frank", "dept_id": 99}, ignore_index=True
) if hasattr(employees, 'append') else pd.concat(
    [employees, pd.DataFrame([{"emp_id":6,"name":"Frank","dept_id":99}])],
    ignore_index=True
)
print("
Left join (Frank has no dept — shows NaN):")
print(employees_new.merge(departments, on="dept_id", how="left"))

## 11.8 Reading and Writing Data

**Pandas can read from almost anywhere:**
CSV files, Excel spreadsheets, JSON, SQL databases, HTML tables, Parquet files (big data format), and more. The read functions all follow the same pattern: `pd.read_XXX(path_or_url)`.

**Best practices when loading:**
- Always check `.shape` and `.head()` immediately after loading
- Use `dtype=` to specify column types upfront if you know them
- Use `parse_dates=` for date columns
- Use `usecols=` to load only the columns you need (saves memory for large files)

In [ ]:
import pandas as pd
from io import StringIO

# Simulating reading a CSV (in practice: df = pd.read_csv("path/to/file.csv"))
csv_data = (
    "name,age,salary,join_date
"
    "Alice,25,55000,2021-03-15
"
    "Bob,30,72000,2019-07-20
"
    "Carol,28,61000,2022-01-10
"
    "David,35,85000,2015-11-05"
)

df = pd.read_csv(StringIO(csv_data), parse_dates=["join_date"])
print(df)
print()
print("Join date dtype:", df["join_date"].dtype)

# Calculate how long each person has been at the company
df["tenure_days"] = (pd.Timestamp.now() - df["join_date"]).dt.days
df["tenure_years"] = (df["tenure_days"] / 365).round(1)
print()
print(df[["name", "join_date", "tenure_years"]])

# Write back to CSV
df.to_csv("/tmp/cleaned_employees.csv", index=False)
print("
Saved to /tmp/cleaned_employees.csv")

---

## Key Takeaways

- A **Series** is a labelled 1D array. A **DataFrame** is a table of Series.
- **Always explore first:** `.shape`, `.dtypes`, `.head()`, `.describe()`, `.isnull().sum()`
- `.loc[]` selects by label; `.iloc[]` selects by position number
- Boolean filtering `df[df["col"] > value]` is the pandas equivalent of SQL WHERE
- **groupby** follows split-apply-combine; use `.agg()` for multiple stats, `.transform()` to add group stats back to each row
- **merge** is SQL JOIN; `how="left"` keeps all rows from the left table
- Pandas can read CSV, Excel, JSON, SQL and many more formats

## Exercises

[11_exercises.ipynb](exercises/11_exercises.ipynb) | [11_solutions.ipynb](exercises/11_solutions.ipynb)

## Next: [12 — Visualisation](../12_visualization/12_visualization.ipynb)
